# Chapter 13 Companion Notebook: Algorithmic Trading and Market Microstructure

This notebook reproduces every worked numerical example from Chapter 13 of *AI in Finance*: the moving-average crossover backtest, a feature-engineering catalog for trading signals (momentum, realized volatility, RSI, APO, Bollinger Bands), a head-to-head test of gradient boosting versus logistic regression on a simulated momentum/volatility-regime interaction, TWAP/VWAP execution, the Almgren-Chriss optimal execution trajectory, Kyle's lambda, the Avellaneda-Stoikov inventory model, the pairs-trading backtest, the multi-stock statistical arbitrage example, backtest performance metrics, and implementation shortfall.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Moving-average crossover strategy (Section 13.2.2)

In [1]:
import numpy as np
import pandas as pd

prices = pd.Series([100,101,103,106,110,115,119,118,114,109,104,100], name='Price')
returns = prices.pct_change()

ma_short = prices.rolling(3).mean()
ma_long = prices.rolling(6).mean()
signal = (ma_short > ma_long).astype(int)
signal_lag = signal.shift(1).fillna(0)

strategy_returns = (signal_lag * returns).fillna(0)

table = pd.DataFrame({
    'Price': prices, 'MA3': ma_short.round(2), 'MA6': ma_long.round(2),
    'Signal': signal, 'Return': returns.round(4), 'StratReturn': strategy_returns.round(4),
})
print(table)

cum_strategy = (1 + strategy_returns).cumprod()
cum_buyhold = (1 + returns.fillna(0)).cumprod()
print(f"\nFinal strategy cumulative return: {cum_strategy.iloc[-1]-1:.4%}")
print(f"Final buy-and-hold cumulative return: {cum_buyhold.iloc[-1]-1:.4%}")

    Price     MA3     MA6  Signal  Return  StratReturn
0     100     NaN     NaN       0     NaN       0.0000
1     101     NaN     NaN       0  0.0100       0.0000
2     103  101.33     NaN       0  0.0198       0.0000
3     106  103.33     NaN       0  0.0291       0.0000
4     110  106.33     NaN       0  0.0377       0.0000
5     115  110.33  105.83       1  0.0455       0.0000
6     119  114.67  109.00       1  0.0348       0.0348
7     118  117.33  111.83       1 -0.0084      -0.0084
8     114  117.00  113.67       1 -0.0339      -0.0339
9     109  113.67  114.17       0 -0.0439      -0.0439
10    104  109.00  113.17       0 -0.0459      -0.0000
11    100  104.33  110.67       0 -0.0385      -0.0000

Final strategy cumulative return: -5.2174%
Final buy-and-hold cumulative return: 0.0000%


## 1b. A feature-engineering catalog for trading signals (Section 13.7.2, Table 13.7)

Reuses the identical twelve-day price series from the moving-average crossover example above.

In [2]:
prices_arr = prices.to_numpy()
rets_arr = prices_arr[1:] / prices_arr[:-1] - 1     # rets_arr[i] realized on day i+2
changes = np.diff(prices_arr)                        # changes[i] realized on day i+2
mom3 = prices_arr[3:] / prices_arr[:-3] - 1           # mom3[i] available as of day i+4

rows = []
for day in range(4, 13):
    p = prices_arr[day - 1]
    m = mom3[day - 4] * 100
    vol_window = rets_arr[day - 4:day - 1]
    v = vol_window.std(ddof=0) * 100
    ch_window = changes[day - 4:day - 1]
    gains = ch_window[ch_window > 0].sum()
    losses = -ch_window[ch_window < 0].sum()
    avg_gain, avg_loss = gains / 3, losses / 3
    rsi = 100.0 if avg_loss == 0 else 100 - 100 / (1 + avg_gain / avg_loss)
    rows.append([day, p, round(m, 2), round(v, 3), round(rsi, 2)])

feat_table = pd.DataFrame(rows, columns=['Day', 'Price', 'Mom3(%)', 'Vol3(%)', 'RSI3'])
print(feat_table.to_string(index=False))

 Day  Price  Mom3(%)  Vol3(%)   RSI3
   4    106     6.00    0.781 100.00
   5    110     8.91    0.732 100.00
   6    115    11.65    0.667 100.00
   7    119    12.26    0.450 100.00
   8    118     7.27    2.328  90.00
   9    114    -0.87    2.835  44.44
  10    109    -8.40    1.493   0.00
  11    104   -11.86    0.523   0.00
  12    100   -12.28    0.313   0.00


## 1c. Absolute Price Oscillator and Bollinger Bands (Section 13.7.2, Table 13.8)

Reuses the identical twelve-day price series, with EMA/SMA windows scaled down from industry-standard lengths to fit the short illustrative series.

In [3]:
import math
import statistics as stats

# Absolute Price Oscillator: fast=3, slow=6 EMA (scaled down from the industry-standard 10/40)
n_fast, n_slow = 3, 6
lam_fast, lam_slow = 2.0/(n_fast+1), 2.0/(n_slow+1)
ema_fast = ema_slow = None
apo_rows = []
for day, p in enumerate(prices_arr, start=1):
    if ema_fast is None:
        ema_fast = ema_slow = p
    else:
        ema_fast = lam_fast*p + (1-lam_fast)*ema_fast
        ema_slow = lam_slow*p + (1-lam_slow)*ema_slow
    apo_rows.append([day, p, round(ema_fast,3), round(ema_slow,3), round(ema_fast-ema_slow,3)])

apo_table = pd.DataFrame(apo_rows, columns=['Day','Price','EMA3','EMA6','APO'])
print(apo_table.to_string(index=False))

 Day  Price    EMA3    EMA6    APO
   1    100 100.000 100.000  0.000
   2    101 100.500 100.286  0.214
   3    103 101.750 101.061  0.689
   4    106 103.875 102.472  1.403
   5    110 106.938 104.623  2.314
   6    115 110.969 107.588  3.381
   7    119 114.984 110.849  4.136
   8    118 116.492 112.892  3.600
   9    114 115.246 113.208  2.038
  10    109 112.123 112.006  0.117
  11    104 108.062 109.719 -1.657
  12    100 104.031 106.942 -2.911


In [4]:
# Bollinger Bands: 5-day SMA, k=2 (scaled down from the industry-standard 20-day)
# note: statistics.mean() silently truncates to numpy's integer dtype when given
# numpy.int64 values (unlike Python's built-in int), so prices are cast to float first
num_periods, stdev_factor = 5, 2.0
history = []
bb_rows = []
for day, p in enumerate(prices_arr, start=1):
    history.append(float(p))
    if len(history) > num_periods:
        del history[0]
    sma = stats.mean(history)
    variance = sum((h-sma)**2 for h in history)
    stdev = math.sqrt(variance/len(history))
    bb_rows.append([day, p, round(sma,3), round(sma+stdev_factor*stdev,3), round(sma-stdev_factor*stdev,3)])

bb_table = pd.DataFrame(bb_rows, columns=['Day','Price','Middle(SMA5)','Upper','Lower'])
print(bb_table.to_string(index=False))

 Day  Price  Middle(SMA5)   Upper   Lower
   1    100       100.000 100.000 100.000
   2    101       100.500 101.500  99.500
   3    103       101.333 103.828  98.839
   4    106       102.500 107.083  97.917
   5    110       104.000 111.266  96.734
   6    115       107.000 117.040  96.960
   7    119       110.600 122.234  98.966
   8    118       113.600 123.447 103.753
   9    114       115.200 121.575 108.825
  10    109       115.000 122.043 107.957
  11    104       112.800 124.071 101.529
  12    100       109.000 122.023  95.977


## 1d. Machine learning for trading signals: gradient boosting vs. logistic regression (Section 13.7.1)

The twelve-day price series above is too short to fit any model. Here we simulate 5,000 days with a genuine, realistic interaction: 3-day momentum continues (trend-following works) in calm, low-volatility regimes but reverses (mean-reversion dominates) in turbulent, high-volatility regimes, a nonlinear interaction between momentum and volatility that a logistic regression with only additive main effects cannot represent. Both models are evaluated on a genuinely held-out final segment of the series, never touched during training, consistent with the walk-forward discipline Chapter 7 developed for time-ordered data.

In [5]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng_sig = np.random.default_rng(9)
n_days = 5_000

sim_returns = np.zeros(n_days)
sim_sigma2 = np.zeros(n_days)
sim_sigma2[0] = 0.0001
sim_returns[0] = rng_sig.normal(0, np.sqrt(sim_sigma2[0]))
for t in range(1, n_days):
    sim_sigma2[t] = 0.00001 + 0.08*sim_returns[t-1]**2 + 0.85*sim_sigma2[t-1]
    sim_returns[t] = rng_sig.normal(0, np.sqrt(sim_sigma2[t]))
sim_prices = 100 * np.cumprod(1 + sim_returns)

sig_df = pd.DataFrame({"price": sim_prices, "ret": sim_returns})
sig_df["mom3"] = sig_df["price"].pct_change(3)
sig_df["vol5"] = sig_df["ret"].rolling(5).std()
delta_sig = sig_df["price"].diff()
gain_sig = delta_sig.clip(lower=0).rolling(3).mean()
loss_sig = (-delta_sig.clip(upper=0)).rolling(3).mean()
sig_df["rsi3"] = 100 - 100 / (1 + gain_sig / loss_sig.replace(0, np.nan))
sig_df["rsi3"] = sig_df["rsi3"].fillna(50)
sig_df = sig_df.dropna().reset_index(drop=True)

# True rule: momentum continues in calm regimes, reverses in turbulent regimes --
# a fixed volatility threshold defines the regime, not the sample's own distribution.
vol_threshold = 0.01
high_vol = (sig_df["vol5"] > vol_threshold).astype(float)
z_sig = 8.0 * sig_df["mom3"] * (1 - high_vol) - 8.0 * sig_df["mom3"] * high_vol
p_up = 1 / (1 + np.exp(-z_sig))
sig_df["target"] = rng_sig.binomial(1, p_up.values)
print(f"Fraction of days in the high-volatility regime: {high_vol.mean():.3f}")
print(f"Base up-rate: {sig_df['target'].mean():.4f}")

n_train_sig = 3_500
train_sig, test_sig = sig_df.iloc[:n_train_sig], sig_df.iloc[n_train_sig:]
sig_features = ["mom3", "vol5", "rsi3"]
X_train_sig, y_train_sig = train_sig[sig_features].values, train_sig["target"].values
X_test_sig, y_test_sig = test_sig[sig_features].values, test_sig["target"].values
print(f"n_train={len(train_sig)}, n_test={len(test_sig)} (held out, never seen during training)")

logit_sig = LogisticRegression().fit(X_train_sig, y_train_sig)
auc_logit_sig = roc_auc_score(y_test_sig, logit_sig.predict_proba(X_test_sig)[:, 1])
print(f"\nLogistic regression held-out AUC: {auc_logit_sig:.4f}")

gbm_sig = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=0)
gbm_sig.fit(X_train_sig, y_train_sig)
auc_gbm_sig = roc_auc_score(y_test_sig, gbm_sig.predict_proba(X_test_sig)[:, 1])
print(f"Gradient boosting held-out AUC: {auc_gbm_sig:.4f}")
print(f"Improvement: {auc_gbm_sig - auc_logit_sig:.4f}")
print(f"GBM feature importances (mom3, vol5, rsi3): {gbm_sig.feature_importances_}")

Fraction of days in the high-volatility regime: 0.569
Base up-rate: 0.5026
n_train=3500, n_test=1496 (held out, never seen during training)

Logistic regression held-out AUC: 0.5130


Gradient boosting held-out AUC: 0.5328
Improvement: 0.0198
GBM feature importances (mom3, vol5, rsi3): [0.42275537 0.387809   0.18943563]


## 2. Order book walk and TWAP/VWAP (Section 13.3.1, Section 13.3.4)

In [6]:
order_book_asks = pd.DataFrame({'Price': [50.10, 50.15, 50.20], 'Size': [100, 200, 300]}, index=[1,2,3])
best_ask = 50.10
order_size = 250
filled, cost = 0, 0.0
for price, size in zip(order_book_asks['Price'], order_book_asks['Size']):
    take = min(size, order_size - filled)
    cost += take * price
    filled += take
    if filled >= order_size:
        break
avg_price = cost / order_size
print(f"Aggressive execution avg price: ${avg_price:.2f}, impact: {(avg_price-best_ask)/best_ask*10000:.1f} bps")
print(f"Patient (TWAP-style) execution avg price: ${best_ask:.2f}, impact: 0.0 bps")

# TWAP/VWAP on Table 11.3
twap_prices = np.array([50.00, 50.05, 50.10, 50.08, 50.12])
twap_volumes = np.array([10_000, 12_000, 8_000, 15_000, 9_000])
twap_price = twap_prices.mean()
vwap_price = np.sum(twap_prices * twap_volumes) / np.sum(twap_volumes)
print(f"\nTWAP execution price: ${twap_price:.4f}")
print(f"VWAP benchmark: ${vwap_price:.4f}")
print(f"Difference: {(twap_price-vwap_price)/vwap_price*10000:.2f} bps")

Aggressive execution avg price: $50.13, impact: 6.0 bps
Patient (TWAP-style) execution avg price: $50.10, impact: 0.0 bps

TWAP execution price: $50.0700
VWAP benchmark: $50.0681
Difference: 0.37 bps


## 2b. The Almgren-Chriss optimal execution trajectory (Section 13.3.6)

Split the same 250-share order across N=5 intervals using the closed-form Almgren-Chriss trajectory, and compare its risk-adjusted cost to the uniform (TWAP) and fully aggressive extremes.

In [7]:
X_ac, N_ac, tau_ac = 250.0, 5, 1.0
sigma_ac, eta_ac = 0.02, 0.01
T_ac = N_ac * tau_ac
t_j = np.arange(0, N_ac + 1) * tau_ac

def ac_trajectory(lam):
    kappa = np.sqrt(lam * sigma_ac**2 / eta_ac)
    return kappa, X_ac * np.sinh(kappa * (T_ac - t_j)) / np.sinh(kappa * T_ac)

def risk_adjusted_cost(x, lam):
    dx = -np.diff(x)
    E_cost = eta_ac * np.sum(dx**2) / tau_ac
    Var_cost = sigma_ac**2 * tau_ac * np.sum(x[1:]**2)
    return E_cost, Var_cost, E_cost + lam * Var_cost

lam_low = 0.5
kappa_low, x_opt_low = ac_trajectory(lam_low)
print(f"lambda={lam_low}: kappa={kappa_low:.4f}")
print(f"  remaining shares: {x_opt_low.round(2)}")
print(f"  traded per interval: {(-np.diff(x_opt_low)).round(2)}")

lam_high = 5.0
kappa_high, x_opt_high = ac_trajectory(lam_high)
print(f"\nlambda={lam_high}: kappa={kappa_high:.4f}")
print(f"  remaining shares: {x_opt_high.round(2)}")
print(f"  traded per interval: {(-np.diff(x_opt_high)).round(2)}")

# Compare three strategies at lambda=0.5: optimal, uniform (TWAP), fully aggressive
x_uniform = np.linspace(X_ac, 0, N_ac + 1)
x_aggressive = np.array([X_ac] + [0.0] * N_ac)

for label, x in [("Optimal (AC)", x_opt_low), ("Uniform (TWAP)", x_uniform), ("Aggressive", x_aggressive)]:
    E_c, Var_c, obj = risk_adjusted_cost(x, lam_low)
    print(f"\n{label}: E[cost]={E_c:.1f}, Var[cost]={Var_c:.1f}, objective={obj:.1f}")

lambda=0.5: kappa=0.1414
  remaining shares: [250.   194.24 142.38  93.36  46.22   0.  ]
  traded per interval: [55.76 51.87 49.01 47.14 46.22]

lambda=5.0: kappa=0.4472
  remaining shares: [250.   157.18  96.33  55.06  24.99   0.  ]
  traded per interval: [92.82 60.86 41.27 30.07 24.99]

Optimal (AC): E[cost]=125.6, Var[cost]=27.5, objective=139.4

Uniform (TWAP): E[cost]=125.0, Var[cost]=30.0, objective=140.0

Aggressive: E[cost]=625.0, Var[cost]=0.0, objective=625.0


## 3. Kyle's lambda (Section 13.3.3, Table 13.2)

In [8]:
order_flow = np.array([5, -3, 8, -6, 2, 4], dtype=float)
price_change = np.array([10, -7, 15, -11, 3, 9], dtype=float)

xbar, ybar = order_flow.mean(), price_change.mean()
lam = np.sum((order_flow - xbar) * (price_change - ybar)) / np.sum((order_flow - xbar) ** 2)
intercept = ybar - lam * xbar
fitted = intercept + lam * order_flow
r2 = 1 - np.sum((price_change - fitted) ** 2) / np.sum((price_change - ybar) ** 2)

print(f"Kyle's lambda: {lam:.4f} cents per thousand shares")
print(f"Intercept: {intercept:.4f}")
print(f"R^2: {r2:.4f}")

Kyle's lambda: 1.9466 cents per thousand shares
Intercept: -0.0777
R^2: 0.9915


## 4. Avellaneda-Stoikov inventory model (Section 13.4)

In [9]:
s, q, gamma, sigma, T_t, kappa = 100.0, 5, 0.1, 2.0, 1.0, 1.5

reservation_price = s - q * gamma * sigma**2 * T_t
as_spread = gamma * sigma**2 * T_t + (2/gamma) * np.log(1 + gamma/kappa)
bid = reservation_price - as_spread/2
ask = reservation_price + as_spread/2

sym_bid = s - as_spread/2
sym_ask = s + as_spread/2

print(f"Reservation price: ${reservation_price:.2f}")
print(f"Optimal spread: ${as_spread:.4f}")
print(f"Inventory-aware bid/ask: ${bid:.2f} / ${ask:.2f}")
print(f"Symmetric (no-inventory) bid/ask: ${sym_bid:.2f} / ${sym_ask:.2f}")
print(f"Skew: ${s - reservation_price:.2f}")

Reservation price: $98.00
Optimal spread: $1.6908
Inventory-aware bid/ask: $97.15 / $98.85
Symmetric (no-inventory) bid/ask: $99.15 / $100.85
Skew: $2.00


## 5. Pairs-trading backtest (Section 13.5.1, Table 13.4)

In [10]:
A = np.array([50,51,53,52,54,55,58,56.5,54.8,54.5])
B = np.array([48,49,50,50,51,52,54,53,52,52])
spread = A - B
print(f"Spread: {spread}")

mu_in = spread[:6].mean()
sigma_in = spread[:6].std(ddof=1)
z = (spread - mu_in) / sigma_in
print(f"In-sample mean={mu_in:.2f}, std={sigma_in:.4f}")
print(f"Out-of-sample z-scores (days 7-10): {z[6:].round(2)}")

entry_idx, exit_idx = 6, 9
pnl = spread[entry_idx] - spread[exit_idx]
capital = A[entry_idx] + B[entry_idx]
print(f"\nEntry spread: {spread[entry_idx]:.2f}, Exit spread: {spread[exit_idx]:.2f}")
print(f"P&L per share-pair: ${pnl:.2f}")
print(f"Return on capital: {pnl/capital:.4%}")

Spread: [2.  2.  3.  2.  3.  3.  4.  3.5 2.8 2.5]
In-sample mean=2.50, std=0.5477
Out-of-sample z-scores (days 7-10): [2.74 1.83 0.55 0.  ]

Entry spread: 4.00, Exit spread: 2.50
P&L per share-pair: $1.50
Return on capital: 1.3393%


## 6. Statistical arbitrage beyond pairs (Section 13.5.2, Table 13.5)

In [11]:
market = np.array([-2.0,-1.0,0.0,1.0,2.0,3.0])
beta_W, beta_X, beta_Y = 1.0, 1.2, 0.8
resid_W = np.array([0.10,-0.10,0.00,0.10,-0.10, 1.20])
resid_X = np.array([0.00, 0.10,-0.10,0.00, 0.10,-0.10])
resid_Y = np.array([-0.10,0.00, 0.10,-0.10,0.00, 0.10])

W = beta_W*market + resid_W
X = beta_X*market + resid_X
Y = beta_Y*market + resid_Y

def estimate_beta(m, r):
    mbar, rbar = m.mean(), r.mean()
    b = np.sum((m-mbar)*(r-rbar))/np.sum((m-mbar)**2)
    a = rbar - b*mbar
    return a, b

for name, series in [('W', W), ('X', X), ('Y', Y)]:
    m_in, r_in = market[:5], series[:5]
    a, b = estimate_beta(m_in, r_in)
    resid_in = r_in - (a + b*m_in)
    resid_std = resid_in.std(ddof=1)
    resid_6 = series[5] - (a + b*market[5])
    z6 = (resid_6 - resid_in.mean())/resid_std
    print(f"{name}: beta={b:.2f}, month-6 residual={resid_6:.2f}%, z-score={z6:.1f}")

W: beta=0.98, month-6 residual=1.26%, z-score=13.3
X: beta=1.21, month-6 residual=-0.15%, z-score=-1.8
Y: beta=0.81, month-6 residual=0.09%, z-score=1.1


## 7. Backtest performance metrics (Section 13.6.1)

In [12]:
# Twelve prices give eleven returns: day 1 has no prior price, so it is not a return period.
strat_returns_full = pd.Series([0.0,0.0,0.0,0.0,0.0,0.0348,-0.0084,-0.0339,-0.0439,-0.0,-0.0])
cum = (1 + strat_returns_full).cumprod()
running_max = cum.cummax()
drawdown = cum / running_max - 1
max_dd = drawdown.min()

nonzero_days = strat_returns_full[strat_returns_full != 0]
hit_rate = (nonzero_days > 0).mean()

mean_r, std_r = strat_returns_full.mean(), strat_returns_full.std(ddof=1)
sharpe = mean_r / std_r

print(f"Max drawdown: {max_dd:.4%}")
print(f"Hit rate (active days): {hit_rate:.0%}")
print(f"Sharpe ratio: {sharpe:.2f}")

Max drawdown: -8.4071%
Hit rate (active days): 25%
Sharpe ratio: -0.23


## 7b. The cost of searching: backtest overfitting (Section 13.6.3)

Reproduces Table 13.6 and every figure quoted in Section 13.6.3. All names are
prefixed `bo_` so nothing in this cell can clobber a variable an earlier or later
cell depends on.

In [13]:
import numpy as np
from scipy import stats
from scipy.integrate import quad

bo_YEARS = 5                                  # five years of daily data
bo_T = bo_YEARS * 252
bo_SE = 1 / np.sqrt(bo_YEARS)                 # SE of an annualised Sharpe, no-edge case
print(f"SE of annualised Sharpe over {bo_YEARS}y of daily data = 1/sqrt({bo_YEARS}) "
      f"= {bo_SE:.4f}")
print(f"naive one-sided 5% hurdle = 1.645 * {bo_SE:.3f} = {1.645*bo_SE:.3f}\n")

def bo_emax(N):
    """E[max of N iid standard normals], by numerical integration."""
    hi = quad(lambda x: 1 - stats.norm.cdf(x) ** N, 0, 12)[0]
    lo = quad(lambda x: stats.norm.cdf(x) ** N, -12, 0)[0]
    return hi - lo

def bo_crit(N, alpha=0.05):
    """Value the max of N must exceed for a family-wise alpha test."""
    return stats.norm.ppf((1 - alpha) ** (1 / N))

print("Table 13.6 -- best-of-N in-sample Sharpe when every strategy is worthless")
print(f"{'N':>6} {'E[max] z':>9} {'best SR':>9} {'5% hurdle':>10}")
for bo_N in (1, 10, 100, 1000):
    bo_z = bo_emax(bo_N)
    print(f"{bo_N:>6} {bo_z:>9.2f} {bo_z*bo_SE:>9.2f} {bo_crit(bo_N)*bo_SE:>10.2f}")

# --- Monte Carlo: 100 genuinely worthless strategies, in-sample and out ------
bo_N_STRAT, bo_REPS = 100, 10_000
bo_rng = np.random.default_rng(13)

def bo_sharpe(r):
    return r.mean(axis=0) / r.std(axis=0, ddof=1) * np.sqrt(252)

bo_is_max = np.empty(bo_REPS)
bo_oos_win = np.empty(bo_REPS)
bo_rho_io = np.empty(bo_REPS)
for bo_i in range(bo_REPS):
    bo_ins = bo_rng.normal(0, 0.01, size=(bo_T, bo_N_STRAT))   # zero true edge
    bo_oos = bo_rng.normal(0, 0.01, size=(bo_T, bo_N_STRAT))
    bo_si, bo_so = bo_sharpe(bo_ins), bo_sharpe(bo_oos)
    bo_w = int(np.argmax(bo_si))                               # pick the winner
    bo_is_max[bo_i] = bo_si[bo_w]
    bo_oos_win[bo_i] = bo_so[bo_w]
    bo_rho_io[bo_i] = stats.spearmanr(bo_si, bo_so).statistic

print(f"\nMonte Carlo: {bo_REPS} runs x {bo_N_STRAT} worthless strategies, "
      f"{bo_YEARS}y in-sample + {bo_YEARS}y out-of-sample")
print(f"  mean best in-sample Sharpe    = {bo_is_max.mean():.3f}   "
      f"(analytic {bo_emax(bo_N_STRAT)*bo_SE:.3f})")
print(f"  mean OOS Sharpe of the winner = {bo_oos_win.mean():+.3f}")
print(f"    90% range                   = [{np.percentile(bo_oos_win,5):+.2f}, "
      f"{np.percentile(bo_oos_win,95):+.2f}]")
print(f"  P(winner loses money OOS)     = {(bo_oos_win<0).mean():.1%}")
print(f"  mean in-sample/OOS rank corr  = {bo_rho_io.mean():+.3f}")

# --- Correlated grid points shrink the effective number of trials -----------
print("\nOverlapping parameter grids are correlated, which shrinks effective N:")
bo_rng2 = np.random.default_rng(1313)
for bo_rho in (0.0, 0.3, 0.5, 0.7, 0.9):
    bo_mx = np.empty(2000)
    for bo_j in range(2000):
        bo_common = bo_rng2.normal(0, 0.01, size=(bo_T, 1))
        bo_idio = bo_rng2.normal(0, 0.01, size=(bo_T, bo_N_STRAT))
        bo_mx[bo_j] = bo_sharpe(np.sqrt(bo_rho)*bo_common
                                + np.sqrt(1-bo_rho)*bo_idio).max()
    bo_target = bo_mx.mean() / bo_SE                 # equivalent independent N
    bo_lo, bo_hi = 1.0, 1e6
    for _ in range(50):
        bo_mid = np.sqrt(bo_lo * bo_hi)
        if bo_emax(int(round(bo_mid))) < bo_target: bo_lo = bo_mid
        else: bo_hi = bo_mid
    print(f"  rho={bo_rho:.1f}: mean best Sharpe {bo_mx.mean():.2f} "
          f"-> effective independent N ~ {round(np.sqrt(bo_lo*bo_hi)):>4d}")

# --- How much history a REAL edge needs to clear the search -----------------
print("\nYears of daily data needed for a true annualised Sharpe of 0.5 to clear"
      " an N-configuration search:")
for bo_N in (1, 100):
    print(f"  N={bo_N:>4}: ({bo_crit(bo_N):.3f}/0.5)^2 = {(bo_crit(bo_N)/0.5)**2:.0f} years")

SE of annualised Sharpe over 5y of daily data = 1/sqrt(5) = 0.4472
naive one-sided 5% hurdle = 1.645 * 0.447 = 0.736

Table 13.6 -- best-of-N in-sample Sharpe when every strategy is worthless
     N  E[max] z   best SR  5% hurdle
     1     -0.00     -0.00       0.74
    10      1.54      0.69       1.15
   100      2.51      1.12       1.47
  1000      3.24      1.45       1.74



Monte Carlo: 10000 runs x 100 worthless strategies, 5y in-sample + 5y out-of-sample
  mean best in-sample Sharpe    = 1.118   (analytic 1.121)
  mean OOS Sharpe of the winner = +0.004
    90% range                   = [-0.73, +0.73]
  P(winner loses money OOS)     = 49.6%
  mean in-sample/OOS rank corr  = +0.001

Overlapping parameter grids are correlated, which shrinks effective N:


  rho=0.0: mean best Sharpe 1.12 -> effective independent N ~   97


  rho=0.3: mean best Sharpe 0.94 -> effective independent N ~   34


  rho=0.5: mean best Sharpe 0.80 -> effective independent N ~   18


  rho=0.7: mean best Sharpe 0.61 -> effective independent N ~    7


  rho=0.9: mean best Sharpe 0.36 -> effective independent N ~    3

Years of daily data needed for a true annualised Sharpe of 0.5 to clear an N-configuration search:
  N=   1: (1.645/0.5)^2 = 11 years
  N= 100: (3.283/0.5)^2 = 43 years


## 8. Implementation shortfall (Section 13.6.4)

In [14]:
decision_price = 50.00
execution_price = 50.07
shortfall_per_share = execution_price - decision_price
shortfall_bps = shortfall_per_share / decision_price * 10_000
order_size = 1000
total_shortfall = shortfall_per_share * order_size

print(f"Implementation shortfall: ${shortfall_per_share:.2f}/share = {shortfall_bps:.0f} bps")
print(f"Total shortfall on {order_size}-share order: ${total_shortfall:.0f}")

Implementation shortfall: $0.07/share = 14 bps
Total shortfall on 1000-share order: $70


## Exercises (match Chapter 13, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [15]:
# Exercise 3: 500-share order, 200 shares at $30.00, rest at $30.05
book_ex3 = [(30.00, 200), (30.05, 400)]
size_ex3, filled_ex3, cost_ex3 = 500, 0, 0.0
for price, size in book_ex3:
    take = min(size, size_ex3 - filled_ex3)
    cost_ex3 += take * price
    filled_ex3 += take
avg_ex3 = cost_ex3 / size_ex3
print(f"Exercise 3 -- avg price=${avg_ex3:.4f}, impact={(avg_ex3-30.00)/30.00*10000:.2f} bps")

# Exercise 4: TWAP/VWAP for a 2,000-share order (same table, ratios unchanged)
print(f"Exercise 4 -- TWAP=${twap_price:.4f}, VWAP=${vwap_price:.4f} (same regardless of order size)")

# Exercise 18: Avellaneda-Stoikov with q=-8 (short position)
q_ex9 = -8
r_ex9 = s - q_ex9*gamma*sigma**2*T_t
bid_ex9 = r_ex9 - as_spread/2
ask_ex9 = r_ex9 + as_spread/2
print(f"Exercise 18 -- reservation price=${r_ex9:.2f}, bid=${bid_ex9:.2f}, ask=${ask_ex9:.2f}")

# Exercise 19: impermanent loss is symmetric in the price ratio
for r_ex19 in (0.5, 2.0):
    il_ex19 = 2*np.sqrt(r_ex19)/(1 + r_ex19) - 1
    print(f"Exercise 19 -- IL(r={r_ex19}) = {il_ex19:.4%}")

# Exercise 26: VWAP under a mistaken volume forecast
P26 = np.array([50.00, 50.05, 50.10, 50.08, 50.12])
V26 = np.array([10_000, 12_000, 8_000, 15_000, 9_000], dtype=float)
w_real = V26 / V26.sum()
w_fc = np.array([0.15, 0.20, 0.25, 0.15, 0.25])
vwap26 = (P26 * V26).sum() / V26.sum()
achieved26 = (P26 * w_fc).sum()
short26 = achieved26 - vwap26
print(f"Exercise 26 -- VWAP={vwap26:.4f}, achieved={achieved26:.4f}")
print(f"Exercise 26 -- shortfall={short26*100:.2f} cents = {short26/vwap26*10000:.2f} bps = ${short26*1000:.2f} on 1,000 shares")
print("Exercise 26 -- per-interval contribution (cents):",
      np.round((w_fc - w_real) * (P26 - vwap26) * 100, 2))


Exercise 3 -- avg price=$30.0300, impact=10.00 bps
Exercise 4 -- TWAP=$50.0700, VWAP=$50.0681 (same regardless of order size)
Exercise 18 -- reservation price=$103.20, bid=$102.35, ask=$104.05
Exercise 19 -- IL(r=0.5) = -5.7191%
Exercise 19 -- IL(r=2.0) = -5.7191%
Exercise 26 -- VWAP=50.0681, achieved=50.0770
Exercise 26 -- shortfall=0.89 cents = 1.77 bps = $8.85 on 1,000 shares
Exercise 26 -- per-interval contribution (cents): [ 0.24  0.04  0.32 -0.15  0.43]
